# 15 — Machine Learning Operations: Evidence-Based Release Gates

## Objective

This Machine Learning Operations (MLOps) audit tracks Supervised Fine-Tuning (SFT), Direct Preference Optimization (DPO), and Group Relative Policy Optimization (GRPO) as distinct research experiments.

Audit the saved experiments from notebooks 00–14, reconstruct dataset and model lineage, compare matched validation predictions, and decide which artifacts are technically valid, scientifically supported, and deployable. **No model training or project final-test scoring occurs here.**

## Methodology

Read existing manifests and prediction files; independently hash local source and model files; compute paired validation diagnostics; run a synthetic local checkpoint smoke test; and write a reproducible research registry under `exploration/runs/mlops_15/`. A research registry entry is not a production deployment. The 104-record PubMedQA final test stays sealed. Previous validation inspection limits the strength of any model-selection claim.

### Implementation 1 — Resolve the research and deployment workspaces

In [1]:
from pathlib import Path

cwd = Path.cwd().resolve()
if (cwd / "exploration" / "notebooks").is_dir():
    ROOT = cwd
elif cwd.name == "notebooks" and cwd.parent.name == "exploration":
    ROOT = cwd.parent.parent
elif cwd.name == "exploration" and (cwd / "notebooks").is_dir():
    ROOT = cwd.parent
else:
    raise FileNotFoundError("Run from the repository root or exploration/notebooks.")
RESEARCH, RUNS, DATA = ROOT / "exploration", ROOT / "exploration" / "runs", ROOT / "exploration" / "data"
DEPLOYMENT = ROOT / "deployment"
OUT = RUNS / "mlops_15"
OUT.mkdir(parents=True, exist_ok=True)
print("Research:", RESEARCH)
print("Deployment:", DEPLOYMENT)
print("MLOps audit outputs:", OUT)

Research: C:\xuexi\Anaconda\BioLLM-PostTraining\exploration
Deployment: C:\xuexi\Anaconda\BioLLM-PostTraining\deployment
MLOps audit outputs: C:\xuexi\Anaconda\BioLLM-PostTraining\exploration\runs\mlops_15


### Implementation 2 — Immutable configuration, source checksums, and split provenance

The configuration hash identifies **this audit protocol**, not an earlier training run. A content fingerprint is recomputed for the small PubMedQA cache; the Med-REFL-DPO Arrow byte hash is checked against its saved manifest. A local checksum cannot prove an unrecorded upstream dataset revision.

In [2]:
import hashlib
import json
import sys
import datasets
from datasets import Dataset

def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

def canonical_hash(value):
    payload = json.dumps(value, sort_keys=True, ensure_ascii=False, separators=(",", ":"))
    return hashlib.sha256(payload.encode("utf-8")).hexdigest()

split = json.loads((DATA / "pubmedqa_split_manifest.json").read_text(encoding="utf-8"))
preference = json.loads((DATA / "med_refl_reasoning_manifest.json").read_text(encoding="utf-8"))
base_protocol = json.loads((RUNS / "predictions_base_validation_manifest.json").read_text(encoding="utf-8"))
config = {
    "audit_version": 1, "seed": split["seed"], "model_id": base_protocol["model_id"],
    "base_revision": base_protocol["model_revision"],
    "pubmedqa_dataset": [split["dataset"], split["config"]],
    "pubmedqa_fingerprint": split["fingerprint"],
    "preference_dataset": [preference["dataset"], preference["config"]],
    "preference_fingerprint": preference["source_content_fingerprint"],
    "evaluation": "validation; Qwen chat template; greedy; max_new_tokens=8; strict yes/no/maybe parser",
    "final_test_policy": "sealed; no prediction or selection in notebook 15",
}
config_digest = canonical_hash(config)
ids = {name: set(values) for name, values in split["ids"].items()}
if [len(ids[name]) for name in ("train", "validation", "test")] != [775, 121, 104]:
    raise ValueError("Unexpected PubMedQA partition counts.")
if any(ids[a] & ids[b] for a, b in (("train", "validation"), ("train", "test"), ("validation", "test"))):
    raise ValueError("Publication IDs overlap between partitions.")

cache = Path(datasets.config.HF_DATASETS_CACHE)
pubmed_files = list(cache.glob("qiaojin___pub_med_qa/pqa_labeled/*/*/pub_med_qa-train.arrow"))
preference_files = list(cache.glob("hani-lab___med-refl-dpo/reasoning_enhancement/*/*/med-refl-dpo-train.arrow"))
source_checks = {"pubmedqa_cache": "missing", "preference_cache": "missing"}
if len(pubmed_files) == 1:
    source = Dataset.from_file(str(pubmed_files[0]))
    canonical_rows = sorted((str(row["pubid"]).strip(), str(row["question"]),
                             "\n".join(s.strip() for s in row["context"]["contexts"] if isinstance(s, str)),
                             str(row["final_decision"]).lower()) for row in source)
    payload = json.dumps(canonical_rows, ensure_ascii=False, separators=(",", ":"))
    actual = hashlib.sha256(payload.encode("utf-8")).hexdigest()
    if actual != split["fingerprint"]:
        raise ValueError("Cached PubMedQA content differs from the frozen source manifest.")
    source_checks["pubmedqa_cache"] = "verified"
if len(preference_files) == 1:
    if sha256_file(preference_files[0]) != preference["cache_arrow_sha256"]:
        raise ValueError("Cached Med-REFL-DPO bytes differ from the source manifest.")
    source_checks["preference_cache"] = "verified"

print("Configuration SHA-256:", config_digest)
print("Source checks:", source_checks)
print("Partition counts:", {name: len(values) for name, values in ids.items()})
print("PubMedQA fingerprint:", split["fingerprint"])
print("Preference fingerprint:", preference["source_content_fingerprint"])
print("Upstream preference revision:", preference.get("source_revision"))

Configuration SHA-256: 04df4ad81be40314a7a0d5e8263defc11c7b14d9b834413777af668546105685
Source checks: {'pubmedqa_cache': 'verified', 'preference_cache': 'verified'}
Partition counts: {'train': 775, 'validation': 121, 'test': 104}
PubMedQA fingerprint: 26f50b35963ad97e7e379add5f7ad1fdb5c82f87e831fa58ee46e4f44f0e9858
Preference fingerprint: 208c8f751fee62038949f1f4b6461f30332b005f6203ddd7c69d03934517f9c8
Upstream preference revision: None


### Results — Source provenance

The split check protects against exact publication-ID overlap and the source hashes detect a changed local cache. The test partition's count and IDs are checked for manifest integrity; **no test labels are read for scoring**. The preference manifest has no verified upstream revision, so reproducibility is limited to the cached content and its recorded hash.

### Implementation 3 — Checkpoint inventory and lineage

The adapter SHA-256 values identify local files. A present file, valid manifest, and completed training step are separate checks; none is a quality endorsement. Missing artifacts remain marked missing.

In [3]:
from safetensors import safe_open

adapters = {}
for name in ("sft", "dpo", "grpo", "reward_model"):
    directory = RUNS / (name + "_adapter" if name != "reward_model" else name)
    required = [directory / "run_manifest.json", directory / "adapter_config.json",
                directory / "adapter_model.safetensors"]
    record = {"name": name, "path": str(directory), "files_present": all(p.exists() for p in required)}
    if record["files_present"]:
        metadata = json.loads(required[0].read_text(encoding="utf-8"))
        peft_config = json.loads(required[1].read_text(encoding="utf-8"))
        with safe_open(str(required[2]), framework="pt", device="cpu") as weights:
            tensor_count = len(list(weights.keys()))
        expected_source = (preference["source_content_fingerprint"] if name == "reward_model"
                           else split["fingerprint"])
        source_key = "source_fingerprint" if name == "reward_model" else "dataset_fingerprint"
        lineage_ok = (metadata.get("model_id") == config["model_id"]
                      and metadata.get("model_revision") == config["base_revision"]
                      and peft_config.get("base_model_name_or_path") == config["model_id"]
                      and metadata.get(source_key) == expected_source)
        if name in ("dpo", "grpo"):
            parent = metadata.get("sft_checkpoint", metadata.get("sft_source"))
            lineage_ok = lineage_ok and Path(parent).resolve() == (RUNS / "sft_adapter").resolve()
        steps = metadata.get("completed_steps", metadata.get("completed_rollout_steps", 0))
        completed = steps > 0 and "completed" in metadata.get("run_status", "")
        record.update({"lineage_valid": bool(lineage_ok), "training_completed": bool(completed),
                       "steps_or_rollouts": steps, "method": metadata.get("method"),
                       "learning_rate": metadata.get("learning_rate"),
                       "adapter_sha256": sha256_file(required[2]), "tensor_count": tensor_count})
        record["hyperparameters"] = {key: metadata[key] for key in
            ("n_train", "max_length", "max_prompt_length", "max_completion_length",
             "gradient_accumulation_steps", "lora_r", "lora_alpha", "beta",
             "beta_kl", "group_size", "learning_rate") if key in metadata}
        logs_path = directory / "log_history.json"
        if logs_path.exists():
            history = json.loads(logs_path.read_text(encoding="utf-8"))
            record["logged_records"] = len(history)
            record["logged_means"] = {key: sum(float(entry[key]) for entry in history
                                                if key in entry) / sum(key in entry for entry in history)
                                      for key in ("loss", "preference_accuracy", "exact_accuracy",
                                                  "valid_format", "mean_reward", "pairwise_accuracy")
                                      if any(key in entry for entry in history)}
        if name == "sft":
            record["training_loss"] = metadata.get("training_loss")
        if name == "reward_model":
            record["exposed_heldout_before"] = metadata.get("heldout_before")
            record["exposed_heldout_after"] = metadata.get("heldout_after")
        if name == "sft":
            chosen = set(metadata["selected_ids"])
            record["train_ids_valid"] = len(chosen) == metadata["n_train"] and chosen <= ids["train"]
        if name == "grpo":
            chosen = set(metadata["selected_ids"])
            record["train_ids_valid"] = (len(chosen) == len(metadata["selected_ids"])
                                         and chosen == set(metadata["used_rollout_ids"])
                                         and chosen <= ids["train"])
    else:
        record.update({"lineage_valid": False, "training_completed": False,
                       "missing": [p.name for p in required if not p.exists()]})
    adapters[name] = record
    print(name, {key: value for key, value in record.items()
                 if key not in ("path", "adapter_sha256")})

reward_revisions = {}
for name in ("reward_model_revision", "reward_model_revision_seed44"):
    path = RUNS / name / "run_manifest.json"
    if not path.exists():
        reward_revisions[name] = {"status": "missing"}
        continue
    item = json.loads(path.read_text(encoding="utf-8"))
    selected = Path(item["selected_checkpoint"])
    reward_revisions[name] = {
        "selected_step": item["selected_step"],
        "selected_file_present": (selected / "adapter_model.safetensors").exists(),
        "new_test_pairs": item["selected_new_test"]["n_pairs"],
        "new_test_accuracy": item["selected_new_test"]["accuracy"],
        "train_accuracy": item["selected_train"]["accuracy"],
    }
print("Reward continuations:", reward_revisions)

sft {'name': 'sft', 'files_present': True, 'lineage_valid': True, 'training_completed': True, 'steps_or_rollouts': 40, 'method': 'LoRA-unquantized', 'learning_rate': 0.0002, 'tensor_count': 96, 'hyperparameters': {'n_train': 200, 'max_length': 768, 'gradient_accumulation_steps': 4, 'lora_r': 8, 'lora_alpha': 16, 'learning_rate': 0.0002}, 'logged_records': 9, 'logged_means': {'loss': 0.34473538547754284}, 'training_loss': 0.3447353854775429, 'train_ids_valid': True}
dpo {'name': 'dpo', 'files_present': True, 'lineage_valid': True, 'training_completed': True, 'steps_or_rollouts': 30, 'method': 'manual pairwise DPO, unquantized base with LoRA', 'learning_rate': 5e-05, 'tensor_count': 96, 'hyperparameters': {'max_length': 1024, 'gradient_accumulation_steps': 4, 'beta': 0.1, 'learning_rate': 5e-05}, 'logged_records': 30, 'logged_means': {'loss': 0.6810691051185132, 'preference_accuracy': 0.575}}
grpo {'name': 'grpo', 'files_present': True, 'lineage_valid': True, 'training_completed': True, 

### Analysis — Training and reward evidence

The SFT run completed 40 optimizer steps on 200 train IDs. DPO completed 30 optimizer steps from SFT; GRPO completed 40 rollouts on 80 train IDs. The original reward adapter completed 25 steps, and two later continuations selected step 45. Their training fit does **not** make them usable reward signals: each selected continuation scored only 9/22 on a newly reserved prompt-group test, compared with 11/22 for the starting reward adapter. The distinct 104-record PubMedQA final test remains sealed.

### Implementation 4 — Matched evaluation and promotion evidence

For candidate and reference models on the same publication IDs, count cases the candidate fixes and breaks. The paired accuracy difference is

$$\Delta_{\mathrm{accuracy}}=\frac{n_{\mathrm{fixed}}-n_{\mathrm{broken}}}{n_{\mathrm{paired}}}.$$

The two-sided exact paired sign test uses only discordant outcomes. It is a small-sample diagnostic, not a complete clinical validation. The majority-label reference and per-class recall are separate gates.

In [4]:
import pandas as pd
from scipy.stats import binomtest
from sklearn.metrics import f1_score

frames, protocols = {}, {}
for name in ("base", "sft", "dpo", "grpo"):
    csv_path = RUNS / "comparison_12" / f"predictions_{name}_validation.csv"
    protocol_path = csv_path.with_name(csv_path.stem + "_manifest.json")
    if not csv_path.exists() or not protocol_path.exists():
        raise FileNotFoundError(f"Missing verified validation evidence for {name}.")
    frame = pd.read_csv(csv_path, dtype={"id": str})
    protocol = json.loads(protocol_path.read_text(encoding="utf-8"))
    if (frame["id"].tolist() != protocol["evaluated_ids"] or len(frame) != len(ids["validation"])
            or set(frame["id"]) != ids["validation"] or frame["id"].duplicated().any()):
        raise ValueError(f"Invalid validation ID evidence for {name}.")
    frames[name], protocols[name] = frame, protocol

reference = frames["base"]
common_keys = ("model_id", "model_revision", "dataset_fingerprint", "source_partition",
               "evaluated_ids", "decoding", "max_items", "seed", "model_dtype")
for name, frame in frames.items():
    if any(protocols[name][key] != protocols["base"][key] for key in common_keys):
        raise ValueError(f"Incompatible inference protocol for {name}.")
    if any(not frame[column].equals(reference[column]) for column in
           ("id", "label", "question", "input_token_count")):
        raise ValueError(f"Mismatched evaluation inputs for {name}.")

def paired(candidate, comparator):
    a = candidate["prediction"].eq(candidate["label"])
    b = comparator["prediction"].eq(comparator["label"])
    gained, lost = int((a & ~b).sum()), int((~a & b).sum())
    return {"gained": gained, "lost": lost,
            "accuracy_delta": (gained - lost) / len(candidate),
            "exact_two_sided_p": float(binomtest(gained, gained + lost, 0.5).pvalue)
            if gained + lost else 1.0}

majority_prediction = reference.copy()
majority_label = reference["label"].value_counts().idxmax()
majority_prediction["prediction"] = majority_label
comparisons = {"sft_vs_base": paired(frames["sft"], frames["base"]),
               "sft_vs_majority": paired(frames["sft"], majority_prediction),
               "dpo_vs_majority": paired(frames["dpo"], majority_prediction),
               "grpo_vs_majority": paired(frames["grpo"], majority_prediction),
               "dpo_vs_sft": paired(frames["dpo"], frames["sft"]),
               "grpo_vs_sft": paired(frames["grpo"], frames["sft"])}

rows = []
for name, frame in frames.items():
    correct = int(frame["prediction"].eq(frame["label"]).sum())
    maybe = frame[frame["label"].eq("maybe")]
    rows.append({"variant": name, "correct": correct, "n": len(frame),
                 "accuracy": correct / len(frame),
                 "macro_f1": float(f1_score(frame["label"], frame["prediction"],
                                            labels=["yes", "no", "maybe"], average="macro")),
                 "valid_format": float(frame["prediction"].isin(["yes", "no", "maybe"]).mean()),
                 "maybe_recall": float(maybe["prediction"].eq("maybe").mean())})
comparison_table = pd.DataFrame(rows)
print(comparison_table.to_string(index=False, float_format=lambda value: f"{value:.4f}"))
print("Majority-label reference:", majority_label, int(majority_prediction["prediction"].eq(reference["label"]).sum()), "/", len(reference))
for label, values in comparisons.items():
    print(label, values)

# The reward gate requires independent ranking evidence and adjudicated preferences.
reward_new_test = [item.get("new_test_accuracy") for item in reward_revisions.values()]
expert_rankings_verified = False  # No independent clinical adjudication exists.
reward_signal_validated = bool(reward_new_test and all(score is not None and score > 0.5
                                                       for score in reward_new_test)
                               and expert_rankings_verified)
print("Reward signal validated for downstream RLHF/PPO:", reward_signal_validated)

variant  correct   n  accuracy  macro_f1  valid_format  maybe_recall
   base       19 121    0.1570    0.1151        1.0000        1.0000
    sft       73 121    0.6033    0.4216        1.0000        0.0000
    dpo       77 121    0.6364    0.4474        1.0000        0.0000
   grpo       76 121    0.6281    0.4084        1.0000        0.0000
Majority-label reference: yes 64 / 121
sft_vs_base {'gained': 71, 'lost': 17, 'accuracy_delta': 0.4462809917355372, 'exact_two_sided_p': 5.1539546966008475e-09}
sft_vs_majority {'gained': 23, 'lost': 14, 'accuracy_delta': 0.0743801652892562, 'exact_two_sided_p': 0.18774155899882317}
dpo_vs_majority {'gained': 25, 'lost': 12, 'accuracy_delta': 0.10743801652892562, 'exact_two_sided_p': 0.04703102743951604}
grpo_vs_majority {'gained': 14, 'lost': 2, 'accuracy_delta': 0.09917355371900827, 'exact_two_sided_p': 0.004180908203125}
dpo_vs_sft {'gained': 4, 'lost': 0, 'accuracy_delta': 0.03305785123966942, 'exact_two_sided_p': 0.125}
grpo_vs_sft {'gained':

### Results — Quality gates

The 121-question matched table reflects notebook 12: base 19, SFT 73, DPO 77, and GRPO 76 correct. The SFT adapter improves on this particularly weak base; it has not established a reliable advantage over the simple majority predictor and misses all 17 maybe cases. DPO's four corrected SFT cases give exact two-sided `p=0.125`; GRPO's mixed gains and losses give `p≈0.664`. Neither demonstrates a dependable additional gain. The reward-model gate is closed because independent preference discrimination did not improve and source rankings lack expert verification.

**Majority-comparison caution.** DPO and GRPO have unadjusted paired majority comparisons below 0.05 in the saved 121-case validation set. Because the same validation set has been inspected repeatedly and several variants were compared, these exploratory signals are not confirmatory promotion evidence. The incremental comparisons against SFT remain inconclusive.

### Implementation 5 — Research registry, promotion criteria, and rollback

The registry distinguishes **artifact integrity**, **completed optimization**, **matched validation evidence**, and **deployment readiness**. Promotion requires all relevant gates; a lower training loss or a loadable checkpoint is insufficient. Rollback below is an explicitly synthetic state-transition exercise. No production registry exists in this repository.

In [5]:
import datetime

model_rows = {row["variant"]: row for row in rows}
registry = {}
for name in ("base", "sft", "dpo", "grpo", "reward_model"):
    artifact = adapters.get(name)
    technical = (protocols["base"]["model_revision"] == config["base_revision"]
                 if name == "base" else bool(artifact and artifact["files_present"]
                                             and artifact["lineage_valid"]
                                             and artifact.get("tensor_count", 0) > 0))
    trained = False if name == "base" else bool(artifact and artifact["training_completed"])
    if name == "sft":
        matched_gain = (comparisons["sft_vs_base"]["accuracy_delta"] > 0
                        and comparisons["sft_vs_base"]["exact_two_sided_p"] < 0.05)
    elif name in ("dpo", "grpo"):
        matched_gain = (comparisons[f"{name}_vs_sft"]["accuracy_delta"] > 0
                        and comparisons[f"{name}_vs_sft"]["exact_two_sided_p"] < 0.05)
    else:
        matched_gain = False
    class_coverage = bool(model_rows[name]["maybe_recall"] > 0) if name in model_rows else None
    majority_gain = bool(comparisons[f"{name}_vs_majority"]["accuracy_delta"] > 0
                         and comparisons[f"{name}_vs_majority"]["exact_two_sided_p"] < 0.05) if name in ("sft", "dpo", "grpo") else None
    # The PubMedQA final test does not apply to the separate preference scorer.
    final_test_complete = None if name == "reward_model" else False
    service_smoke_complete = False
    matched_gain = bool(matched_gain)
    deployment_ready = bool(name != "reward_model" and all((technical, trained, matched_gain,
                        majority_gain is True, class_coverage is True,
                        final_test_complete, service_smoke_complete)))
    blockers = []
    if not technical: blockers.append("artifact or lineage invalid")
    if not trained: blockers.append("no completed post-training run")
    if name != "reward_model" and not matched_gain:
        blockers.append("no demonstrated matched validation gain")
    if name in ("sft", "dpo", "grpo") and not majority_gain:
        blockers.append("no demonstrated gain over majority reference")
    if class_coverage is False: blockers.append("zero maybe-class recall")
    if name == "reward_model" and not reward_signal_validated:
        blockers.append("preference reward signal unvalidated")
    if final_test_complete is False: blockers.append("sealed final test not run")
    if not service_smoke_complete: blockers.append("service smoke not run")
    registry[name] = {"artifact_integrity": technical, "training_completed": trained,
                      "matched_validation_gain": bool(matched_gain),
                      "maybe_class_covered": class_coverage,
                      "majority_unadjusted_signal": majority_gain,
                      "reward_signal_validated": reward_signal_validated if name == "reward_model" else None,
                      "preference_new_test_evaluated": (
                          all(item.get("new_test_pairs", 0) > 0 for item in reward_revisions.values())
                          if name == "reward_model" else None),
                      "final_test_complete": final_test_complete,
                      "service_smoke_complete": service_smoke_complete,
                      "deployment_ready": deployment_ready,
                      "promotion_blockers": blockers,
                      "state": "research_verified" if technical else "missing_or_incompatible"}
    print(name, registry[name])

def rollback_demo(previous, candidate, candidate_healthy):
    """Illustrative transition only; does not alter a live registry."""
    active = candidate if candidate_healthy else previous
    return {"previous": previous, "attempted": candidate, "active_after_check": active,
            "rolled_back": not candidate_healthy}

rollback = rollback_demo("DEMO-stable", "DEMO-candidate", candidate_healthy=False)
assert rollback["active_after_check"] == "DEMO-stable"
print("Illustrative rollback:", rollback)

report = {"configuration": config, "configuration_sha256": config_digest,
          "source_checks": source_checks, "adapters": adapters,
          "reward_revisions": reward_revisions, "comparison": rows,
          "paired_comparisons": comparisons, "registry": registry,
          "production_model": None, "rollback_demonstration": rollback}
for filename, payload in (("experiment_report.json", report), ("registry.json", registry)):
    target = OUT / filename
    target.write_text(json.dumps(payload, indent=2, sort_keys=True) + "\n", encoding="utf-8")
print("Research registry saved:", OUT / "registry.json")
print("Production model:", report["production_model"])

base {'artifact_integrity': True, 'training_completed': False, 'matched_validation_gain': False, 'maybe_class_covered': True, 'majority_unadjusted_signal': None, 'reward_signal_validated': None, 'preference_new_test_evaluated': None, 'final_test_complete': False, 'service_smoke_complete': False, 'deployment_ready': False, 'promotion_blockers': ['no completed post-training run', 'no demonstrated matched validation gain', 'sealed final test not run', 'service smoke not run'], 'state': 'research_verified'}
sft {'artifact_integrity': True, 'training_completed': True, 'matched_validation_gain': True, 'maybe_class_covered': False, 'majority_unadjusted_signal': False, 'reward_signal_validated': None, 'preference_new_test_evaluated': None, 'final_test_complete': False, 'service_smoke_complete': False, 'deployment_ready': False, 'promotion_blockers': ['no demonstrated gain over majority reference', 'zero maybe-class recall', 'sealed final test not run', 'service smoke not run'], 'state': 'resea

### Results - Registry and rollback

All inventoried checkpoints have traceable local research artifacts, but the registry records `deployment_ready=false` for every entry and `production_model=null`. SFT has a matched gain over the weak base; DPO and GRPO have no demonstrated incremental gain over SFT; the reward adapter fails its independent preference gate. The `DEMO-*` rollback output verifies a state-transition rule only and did not change a deployed model.

### Implementation 6 — Validate the exported checkpoint and run a local inference smoke test

Notebook 13 exported the **SFT merged Hugging Face checkpoint**, not a GGUF or Ollama model. A synthetic chat prompt checks that the merged checkpoint loads and generates a stable, nonempty response. This does not score PubMedQA and does not exercise the deployment API.

In [6]:
import os
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, GenerationConfig

export_dir = RUNS / "merged_sft"
export_manifest_path = export_dir / "export_manifest.json"
export_check = {"present": export_manifest_path.exists(), "weight_hash_valid": False,
                "source_adapter_hash_valid": False, "local_inference_smoke": False,
                "service_smoke": False, "ollama_model_verified": False}
if export_check["present"]:
    export = json.loads(export_manifest_path.read_text(encoding="utf-8"))
    export_check["lineage_valid"] = (export["base_model"] == config["model_id"]
                                     and export["base_revision"] == config["base_revision"]
                                     and export["source_dataset_fingerprint"] == split["fingerprint"])
    export_check["weight_hash_valid"] = all(
        (export_dir / filename).exists() and sha256_file(export_dir / filename) == digest
        for filename, digest in export["weight_files_sha256"].items())
    source_weight = RUNS / "sft_adapter" / "adapter_model.safetensors"
    export_check["source_adapter_hash_valid"] = (source_weight.exists()
        and sha256_file(source_weight) == export["source_adapter_sha256"])
    if all(export_check[key] for key in ("lineage_valid", "weight_hash_valid",
                                         "source_adapter_hash_valid")):
        device = "cuda" if torch.cuda.is_available() else "cpu"
        tokenizer = AutoTokenizer.from_pretrained(export_dir, local_files_only=True)
        model = AutoModelForCausalLM.from_pretrained(
            export_dir, local_files_only=True, dtype=torch.float32).to(device).eval()
        prompt = tokenizer.apply_chat_template([
            {"role": "system", "content": "Answer with exactly one label: yes, no, or maybe."},
            {"role": "user", "content": "Question: Is this a synthetic inference smoke test?"},
        ], tokenize=False, add_generation_prompt=True)
        encoded = tokenizer(prompt, return_tensors="pt", add_special_tokens=False).to(device)
        generation = GenerationConfig(do_sample=False, max_new_tokens=8,
                                      pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id,
                                      eos_token_id=tokenizer.eos_token_id)
        with torch.inference_mode():
            first = model.generate(**encoded, generation_config=generation)
            second = model.generate(**encoded, generation_config=generation)
        first_text = tokenizer.decode(first[0, encoded["input_ids"].shape[-1]:], skip_special_tokens=True)
        second_text = tokenizer.decode(second[0, encoded["input_ids"].shape[-1]:], skip_special_tokens=True)
        export_check["local_inference_smoke"] = bool(first_text.strip() and first_text == second_text)
        export_check["synthetic_response"] = first_text
        del model
        if device == "cuda":
            torch.cuda.empty_cache()

deployment_model_files = sorted(p.name for p in (DEPLOYMENT / "model").iterdir())
dockerfile = (DEPLOYMENT / "Dockerfile").read_text(encoding="utf-8")
requirements = [line.strip().lower() for line in
                (DEPLOYMENT / "requirements.txt").read_text(encoding="utf-8").splitlines()
                if line.strip() and not line.lstrip().startswith("#")]
export_check["deployment_model_files"] = deployment_model_files
export_check["adapter_runtime_in_docker"] = (any(line.startswith("torch") for line in requirements)
                                             and any(line.startswith("peft") for line in requirements))
print("Export and smoke checks:", export_check)
print("Docker copies model directory:", "COPY model" in dockerfile)
if os.getenv("RUN_PACKAGE", "0") == "1":
    raise RuntimeError("No checkpoint passes the scientific and service gates; packaging is blocked.")
else:
    print("No deployment files copied; RUN_PACKAGE is disabled.")
(OUT / "deployment_check.json").write_text(json.dumps(export_check, indent=2) + "\n", encoding="utf-8")

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


C:\Users\weiju\anaconda3\envs\torch\lib\site-packages\transformers\integrations\sdpa_attention.py:92: UserWarning: 1Torch was not compiled with flash attention. (Triggered internally at C:\cb\pytorch_1000000000000\work\aten\src\ATen\native\transformers\cuda\sdp_utils.cpp:555.)
  attn_output = torch.nn.functional.scaled_dot_product_attention(


Export and smoke checks: {'present': True, 'weight_hash_valid': True, 'source_adapter_hash_valid': True, 'local_inference_smoke': True, 'service_smoke': False, 'ollama_model_verified': False, 'lineage_valid': True, 'synthetic_response': 'maybe', 'deployment_model_files': ['.gitkeep'], 'adapter_runtime_in_docker': False}
Docker copies model directory: False
No deployment files copied; RUN_PACKAGE is disabled.


327

### Results — Deployment readiness

The merged SFT checkpoint can pass a local synthetic inference smoke test while **deployment readiness remains false**. The current deployment model directory contains no verified exported model, the minimal Docker requirements do not include the adapter stack, and no Ollama model was created from the Hugging Face export. The service's adapter route also has a different raw-prompt/truncation path from the evaluation chat template. A production promotion would require independent model selection, the sealed final test, a compatible serving artifact, and end-to-end service regression tests. Setting `RUN_PACKAGE=1` cannot bypass these gates.

### Implementation 7 — Regression signals and monitoring plan

The saved labeled validation rows permit an **offline** regression check of changed answers and class coverage. This is not production monitoring; no live request traces exist. A future release monitor should record request and model versions, output validity, evidence support, latency percentiles, failures, privacy-safe drift summaries, and human-reviewed outcomes.

In [7]:
regression = {}
for name in ("sft", "dpo", "grpo"):
    comparator = "base" if name == "sft" else "sft"
    result = paired(frames[name], frames[comparator])
    result["changed_answers"] = int(frames[name]["prediction"].ne(frames[comparator]["prediction"]).sum())
    result["invalid_outputs"] = int((~frames[name]["prediction"].isin(["yes", "no", "maybe"])).sum())
    result["maybe_predictions"] = int(frames[name]["prediction"].eq("maybe").sum())
    regression[name] = result
print("Offline validation regression signals:", regression)
monitoring = {"live_telemetry_available": False,
              "proposed_alerts": ["invalid output rate", "missing maybe-class coverage",
                                  "latency p95 and error rate", "evidence support audit",
                                  "model/version mismatch", "task cohort drift"],
              "promotion_blockers": ["no sealed final test", "no verified service smoke",
                                     "zero maybe recall", "repeatedly inspected validation set"]}
print("Monitoring status and unresolved limitations:", monitoring)
(OUT / "regression_and_monitoring.json").write_text(
    json.dumps({"regression": regression, "monitoring": monitoring}, indent=2) + "\n",
    encoding="utf-8")

Offline validation regression signals: {'sft': {'gained': 71, 'lost': 17, 'accuracy_delta': 0.4462809917355372, 'exact_two_sided_p': 5.1539546966008475e-09, 'changed_answers': 119, 'invalid_outputs': 0, 'maybe_predictions': 0}, 'dpo': {'gained': 4, 'lost': 0, 'accuracy_delta': 0.03305785123966942, 'exact_two_sided_p': 0.125, 'changed_answers': 4, 'invalid_outputs': 0, 'maybe_predictions': 0}, 'grpo': {'gained': 12, 'lost': 9, 'accuracy_delta': 0.024793388429752067, 'exact_two_sided_p': 0.6636238098144531, 'changed_answers': 22, 'invalid_outputs': 0, 'maybe_predictions': 0}}
Monitoring status and unresolved limitations: {'live_telemetry_available': False, 'proposed_alerts': ['invalid output rate', 'missing maybe-class coverage', 'latency p95 and error rate', 'evidence support audit', 'model/version mismatch', 'task cohort drift'], 'promotion_blockers': ['no sealed final test', 'no verified service smoke', 'zero maybe recall', 'repeatedly inspected validation set']}


1179

## Results, interpretation, and limits

**Results.** Source-cache checksums and local adapter/export lineage passed; the matched validation counts were 19/73/77/76. The reward-model gate stayed closed, every registry entry had `deployment_ready=false`, `production_model` was null, and service smoke was false.

**Interpretation.** These are verified research artifacts and a local merged-checkpoint smoke test, not an approved model release.

**Limitations.** The final PubMedQA test, live API inference, runtime monitoring, Ollama conversion, and real rollback were not performed.

**Key Observations.** Training completion, artifact integrity, validation evidence, reward-signal validity, and deployment readiness are distinct gates.

## Continuous Integration (CI) and Continuous Deployment (CD) design

**CI on code or data-protocol changes:** validate notebook schema and compile code cells; check mathematical Markdown and prohibited wording; verify manifests and adapter hashes; reproduce metric calculations from saved CSVs; check train/validation/test ID separation; run fast synthetic inference and API error-path tests. Changes to training code require a new versioned run rather than overwriting prior checkpoints.

**Promotion pipeline:** record a candidate, its exact data/model/config hashes, comparator, paired uncertainty, class-wise regressions, and expert review. Freeze selection **before** one authorized final-test pass. Then build a compatible serving image, import an actual model artifact, run container and API smoke tests with the same chat protocol, scan dependencies, stage the release, monitor outcomes, and keep the previous version available for rollback. No CD stage was executed here.

## Key Observations

- **Technically valid** means the files, metadata, lineage, and local load checks agree.
- **Trained** means an optimizer or rollout run actually completed; it does not establish generalization.
- **Validation improvement** must name a comparator. SFT improves on the weak base, while DPO and GRPO do not have convincing incremental gains over SFT. SFT has not shown balanced three-class behavior or a reliable advantage over majority prediction.
- **Reward-signal suitability** requires independent preference discrimination and trustworthy rankings. Notebook 09 does not meet this condition.
- **Deployment readiness** requires a sealed final evaluation, compatible artifact and runtime, service-level smoke/regression checks, and an approved release. No model meets these conditions in the current evidence.

## Q&A

**Why not promote the DPO adapter because it has the highest point accuracy?** Four SFT errors were corrected on the 121-case validation set, but the exact paired test is inconclusive, all 17 maybe cases were missed, and the validation set has been inspected repeatedly.

**Does a successful local merged-model generation prove that the Docker service is ready?** No. It verifies a different inference path; no compatible model has been packaged into the service or exercised through its endpoint.